# Hybrid CNN-5 + CNN-6 → MI-Filter → mRMR → SVM  (May11)

**Pipeline:**
1. Re-extract embeddings from `May11/Dataset` using CNN-5 and CNN-6 **simultaneously** (same image order)
2. Fuse: 256-d + 512-d = **768-d**
3. Two-stage feature selection: **MI pre-filter** (768 → top 200) then **mRMR (MIQ)** (200 → K*)
4. Tune & fit **RBF-SVM** using stratified subsamples for speed
5. Report: accuracy, macro-F1, confusion matrix, classification report, inference time

> **Why CNN-5 + CNN-6 only?** Removing CNN-2 (128-d) reduces the fused dimension from 896-d to 768-d.  
> **Why two-stage mRMR?** Running mRMR on all 768 features takes hours. The MI pre-filter reduces the input to 200 features first — mRMR then runs in ~2-3 min.
> **Why subsampling?** mRMR and C-grid search use 3,000 stratified samples instead of all 10,500 — same quality, much faster.

In [1]:
# ================================================
# Cell 1 — Install Dependencies
# Run once; safe to re-run (pip skips already-installed packages)
# ================================================
# ── Runtime dependency installer ──────────────────────────────
# pymrmr requires Cython + pytz as C-extension build prerequisites;
# --no-build-isolation reuses the already-installed Cython instead
# of spinning up an isolated build env that would miss it.
# umap-learn is optional — Cell 14 (UMAP) skips gracefully if absent.
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])
pip('Cython')
pip('pytz')
pip('--no-build-isolation', 'pymrmr')
pip('umap-learn')
print('Dependencies ready.')

Dependencies ready.


In [2]:
# ================================================
# Cell 2 — Imports & Config
# All paths are set here — only this cell needs
# editing if the project is moved.
# ================================================
# ── Standard library and scientific computing ──────────────────
# os/sys/time/json/warnings/random: I/O, timing, serialisation
# numpy: array maths  |  pandas: tabular results and CSV export
# matplotlib (Agg backend): headless plot generation on the server
import os, sys, time, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm


# ── PyTorch: model loading, inference, transforms ──────────────
# torch / nn / F   : tensor ops, layers, activations
# DataLoader       : batched image loading with pin_memory
# torchvision      : ImageFolder dataset and Resize+ToTensor pipeline
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ── sklearn: feature selection, SVM, evaluation ────────────────
# StandardScaler    : zero-mean unit-variance normalisation before SVM
# KBinsDiscretizer  : required to convert continuous features for mRMR
# SVC               : RBF-kernel SVM classifier
# StratifiedKFold   : balanced folds for C hyperparameter search
# pairwise_distances: used to estimate the median RBF gamma
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import pairwise_distances
import joblib
import umap

# ─── Paths ────────────────────────────────────────────────────────────────────

# ── Paths ──────────────────────────────────────────────────────
BASE = Path('/home/jenarththan/Desktop/FYP')

# May11 rebalanced dataset: 10,500 train / 1,500 val / 3,000 test
# May11 rebalanced split: 10,500 train / 1,500 val / 3,000 test
# Train + Val are merged for the final SVM fit; test stays held-out.
TRAIN_DIR = BASE / 'May11/Dataset/Training'
VAL_DIR   = BASE / 'May11/Dataset/Validation'
TEST_DIR  = BASE / 'May11/Dataset/Testing'

# CNN-5 and CNN-6 trained weights

# Pre-trained checkpoints — loaded frozen; not retrained in this notebook
PTH_CNN5 = BASE / 'May11/Notebooks/Custom/CNN5_May11_Results/best_custom_5cnn_model.pth'
PTH_CNN6 = BASE / 'May11/Notebooks/Custom/CNN6_May11_Results/best_custom_6cnn_model.pth'

# Output directory for this hybrid run

# All artefacts (embeddings, plots, reports, model) go under OUT_DIR
OUT_DIR   = BASE / 'May11/Notebooks/Custom/Hybrid56_May11_Results'
EMB_DIR   = OUT_DIR / 'Embeddings'
PLOTS_DIR = OUT_DIR / 'Plots'
REPS_DIR  = OUT_DIR / 'Reports'
MDLS_DIR  = OUT_DIR / 'Models'
CKPT_DIR  = OUT_DIR / 'Checkpoints'
for d in [EMB_DIR, PLOTS_DIR, REPS_DIR, MDLS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Constants ────────────────────────────────────────────────────────────────

# ── Constants ──────────────────────────────────────────────────
CLASS_NAMES = ['0', '100', '500', '1000', '1500', '2000']  # milling speeds (RPM)
NUM_CLASSES = 6
IMG_SIZE    = 224
# BATCH_SIZE=16 (reduced from 32) to ease GPU memory during extraction
BATCH_SIZE  = 16    # reduced from 32 to ease GPU memory during extraction
SEED        = 42
# WARMUP=20: flush CUDA kernel-launch queue before timing starts
# N_RUNS=200: enough iterations for stable mean/std
WARMUP      = 20
N_RUNS      = 200


# Fix all random seeds for fully reproducible splits, embeddings, SVM
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()


# CUDA required — all embedding extraction and SVM timing uses GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Dataset: Train={TRAIN_DIR}')
print(f'         Val  ={VAL_DIR}')
print(f'         Test ={TEST_DIR}')
print(f'Output : {OUT_DIR}')

Device : cuda
Dataset: Train=/home/jenarththan/Desktop/FYP/May11/Dataset/Training
         Val  =/home/jenarththan/Desktop/FYP/May11/Dataset/Validation
         Test =/home/jenarththan/Desktop/FYP/May11/Dataset/Testing
Output : /home/jenarththan/Desktop/FYP/May11/Notebooks/Custom/Hybrid56_May11_Results


## CNN Model Definitions + Load
CNN-5 produces 256-d embeddings; CNN-6 produces 512-d embeddings.  
Both models are loaded from their May11 trained weights.

In [3]:
# ================================================
# Cell 3 — CNN-5 and CNN-6 Definitions + Load
# ================================================

# ── CNN-5 architecture ─────────────────────────────────────────
# 5 conv blocks → AdaptiveAvgPool → fc1(512→256) embedding
# Reproduced verbatim from CNN5_May11 training notebook so that
# load_state_dict() succeeds without any key remapping.
class Custom5CNN(nn.Module):
    """5-layer CNN — embedding dimension: 256"""
    def __init__(self, num_classes=6, p_drop=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout       = nn.Dropout(p_drop)
        self.fc1           = nn.Linear(512, 256)
        self.fc2           = nn.Linear(256, num_classes)

    def forward(self, x, return_embedding=False):
        x   = self.pool(F.relu(self.conv1(x)))
        x   = self.pool(F.relu(self.conv2(x)))
        x   = self.pool(F.relu(self.conv3(x)))
        x   = self.pool(F.relu(self.conv4(x)))
        x   = self.pool(F.relu(self.conv5(x)))
        x   = self.adaptive_pool(x).view(x.size(0), -1)
        emb = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(emb))
        return (out, emb) if return_embedding else out



# ── CNN-6 architecture ─────────────────────────────────────────
# 6 conv blocks → AdaptiveAvgPool → fc1(1024→512) embedding
# conv6 has NO MaxPool — spatial dims stay 3×3 before GAP.
# Reproduced verbatim from CNN6_May11 training notebook.
class Custom6CNN(nn.Module):
    """6-layer CNN — embedding dimension: 512"""
    def __init__(self, num_classes=6, p_drop=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,    32,   kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32,   64,   kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64,   128,  kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128,  256,  kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256,  512,  kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(512,  1024, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout       = nn.Dropout(p_drop)
        self.fc1           = nn.Linear(1024, 512)
        self.fc2           = nn.Linear(512, num_classes)

    def forward(self, x, return_embedding=False):
        x   = self.pool(F.relu(self.conv1(x)))
        x   = self.pool(F.relu(self.conv2(x)))
        x   = self.pool(F.relu(self.conv3(x)))
        x   = self.pool(F.relu(self.conv4(x)))
        x   = self.pool(F.relu(self.conv5(x)))
        x   = F.relu(self.conv6(x))
        x   = self.adaptive_pool(x).view(x.size(0), -1)
        emb = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(emb))
        return (out, emb) if return_embedding else out



# ── Checkpoint loader ───────────────────────────────────────────
# weights_only=True is preferred (safer); falls back for older PyTorch.
# Also handles checkpoints wrapped as {'model_state_dict': ...}.
def load_model(cls, pth_path, device):
    """Load a model checkpoint — handles both raw state-dict and wrapped formats."""
    model = cls(num_classes=NUM_CLASSES)
    try:
        state = torch.load(str(pth_path), map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(str(pth_path), map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state)
    return model.to(device).eval()


print('Loading CNN-5 and CNN-6 ...')

# ── Load both CNNs in eval mode ────────────────────────────────
# .eval() disables dropout so embeddings are deterministic.
cnn5 = load_model(Custom5CNN, PTH_CNN5, DEVICE)
cnn6 = load_model(Custom6CNN, PTH_CNN6, DEVICE)
print('Models loaded.')


# Parameter and disk-size summary — useful for the thesis model table
rows = []
for name, model, pth, edim in [
    ('CNN-5', cnn5, PTH_CNN5, 256),
    ('CNN-6', cnn6, PTH_CNN6, 512),
]:
    total = sum(p.numel() for p in model.parameters())
    rows.append({
        'Model'           : name,
        'Embedding Dim'   : edim,
        'Total Parameters': f'{total:,}',
        'Disk Size (MB)'  : round(os.path.getsize(str(pth)) / 1e6, 3),
    })
df_models = pd.DataFrame(rows)
print('\n' + df_models.to_string(index=False))
df_models.to_csv(REPS_DIR / 'model_parameter_summary.csv', index=False)

Loading CNN-5 and CNN-6 ...
Models loaded.

Model  Embedding Dim Total Parameters  Disk Size (MB)
CNN-5            256        1,701,446           6.811
CNN-6            512        6,816,070          27.270


## Extract Embeddings (CNN-5 + CNN-6 — Same Image Order)
Both CNNs process the **exact same batch** of images in sequence.  
`shuffle=False` guarantees row alignment across embeddings.  
Train, Validation, and Test splits are extracted and cached separately.  
Train + Val are merged for the final SVM fit; only Test is held out for evaluation.

In [4]:
# ================================================
# Cell 4 — Extract Embeddings
# CNN-5 (256-d) + CNN-6 (512-d) = 768-d fused
# Cached to Embeddings/aligned_56_*.npz
# ================================================
# ── Embedding cache ────────────────────────────────────────────
# Extraction takes ~3 min on the 3050Ti; cache prevents re-running.
# All three splits are cached separately and later merged for SVM.
CACHE_TR  = EMB_DIR / 'aligned_56_train.npz'
CACHE_VAL = EMB_DIR / 'aligned_56_val.npz'
CACHE_TE  = EMB_DIR / 'aligned_56_test.npz'

# Load from cache if all three split files already exist
if CACHE_TR.exists() and CACHE_VAL.exists() and CACHE_TE.exists():
    print('Loading cached embeddings (CNN-5 + CNN-6)...')
    tr_data = np.load(CACHE_TR);  va_data = np.load(CACHE_VAL);  te_data = np.load(CACHE_TE)
    emb5_train, emb6_train = tr_data['e5'], tr_data['e6']
    emb5_val,   emb6_val   = va_data['e5'], va_data['e6']
    emb5_te,    emb6_te    = te_data['e5'], te_data['e6']
    y_train, y_val, y_te   = tr_data['y'],  va_data['y'],  te_data['y']
else:
    # ── Inference transform ─────────────────────────────────────────
    # Resize + ToTensor only — NO ImageNet normalisation.
    # Custom CNNs were trained without normalisation, so omitting it
    # here keeps the embedding distribution consistent with training.
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])
    train_ds = datasets.ImageFolder(str(TRAIN_DIR), transform=transform)
    val_ds   = datasets.ImageFolder(str(VAL_DIR),   transform=transform)
    test_ds  = datasets.ImageFolder(str(TEST_DIR),  transform=transform)

    # ImageFolder sorts class folders alphabetically; remap to our numeric order
    # ── Class-order fix ────────────────────────────────────────────
    # ImageFolder sorts class folders alphabetically, giving
    # '0','1000','100','1500','2000','500' — remap to numeric order.
    alpha_to_num = {old: CLASS_NAMES.index(cls)
                    for cls, old in train_ds.class_to_idx.items()}
    for ds in [train_ds, val_ds, test_ds]:
        ds.targets  = [alpha_to_num[t] for t in ds.targets]
        ds.samples  = [(s, alpha_to_num[t]) for s, t in ds.samples]
        ds.class_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}


    # shuffle=False is critical: ensures CNN-5 and CNN-6 embeddings
    # correspond to the same image on every row (alignment guarantee).
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')


    # ── Embedding extraction ────────────────────────────────────────
    # Passes each batch through CNN-5 and CNN-6 simultaneously so
    # row i of emb5 and emb6 always comes from the same image.
    def extract_all(loader, desc):
        e5_list, e6_list, y_list = [], [], []
        for imgs, lbls in tqdm(loader, desc=desc):
            imgs = imgs.to(DEVICE)
            with torch.no_grad():
                _, e5 = cnn5(imgs, return_embedding=True)
                _, e6 = cnn6(imgs, return_embedding=True)
            e5_list.append(e5.cpu().numpy())
            e6_list.append(e6.cpu().numpy())
            y_list.append(lbls.numpy())
        return (np.concatenate(e5_list), np.concatenate(e6_list),
                np.concatenate(y_list))

    print('\nExtracting train embeddings...')
    emb5_train, emb6_train, y_train = extract_all(train_loader, 'Train')
    print('Extracting validation embeddings...')
    emb5_val, emb6_val, y_val = extract_all(val_loader, 'Validation')
    print('Extracting test embeddings...')
    emb5_te, emb6_te, y_te = extract_all(test_loader, 'Test')

    np.savez_compressed(CACHE_TR,  e5=emb5_train, e6=emb6_train, y=y_train)
    np.savez_compressed(CACHE_VAL, e5=emb5_val,   e6=emb6_val,   y=y_val)
    np.savez_compressed(CACHE_TE,  e5=emb5_te,    e6=emb6_te,    y=y_te)
    print('Cached → Embeddings/')

# Merge train + val for SVM fitting (test stays held-out)

# Merge train + val into one pool for the final SVM fit.
# The test split (3,000 images) stays held-out throughout.
emb5_tr = np.concatenate([emb5_train, emb5_val], axis=0)
emb6_tr = np.concatenate([emb6_train, emb6_val], axis=0)
y_tr    = np.concatenate([y_train, y_val], axis=0)

print(f'\nShapes: CNN-5={emb5_tr.shape}  CNN-6={emb6_tr.shape}')
print(f'Labels: train={y_train.shape}  val={y_val.shape}  train+val={y_tr.shape}  test={y_te.shape}')

Train: 10500  |  Val: 1500  |  Test: 3000

Extracting train embeddings...


Train:   0%|          | 0/657 [00:00<?, ?it/s]

Extracting validation embeddings...


Validation:   0%|          | 0/94 [00:00<?, ?it/s]

Extracting test embeddings...


Test:   0%|          | 0/188 [00:00<?, ?it/s]

Cached → Embeddings/

Shapes: CNN-5=(12000, 256)  CNN-6=(12000, 512)
Labels: train=(10500,)  val=(1500,)  train+val=(12000,)  test=(3000,)


## Per-Image CNN Inference Time
Measures how long CNN-5 + CNN-6 take together to produce embeddings for one image.

In [5]:
# ================================================
# Cell 5 — CNN Extraction Inference Time
# ================================================
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# WARMUP passes flush the CUDA kernel-launch queue; timings after
# warmup reflect steady-state GPU throughput, not cold-start cost.
for _ in range(WARMUP):
    with torch.no_grad():
        cnn5(dummy, return_embedding=True)
        cnn6(dummy, return_embedding=True)


# Time both CNNs together — this is the extraction stage of the pipeline
cnn_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        cnn5(dummy, return_embedding=True)
        cnn6(dummy, return_embedding=True)
    cnn_times.append((time.perf_counter() - t0) * 1000)

cnn_mean_ms = float(np.mean(cnn_times))
cnn_std_ms  = float(np.std(cnn_times))
print(f'CNN-5+6 extraction per image: {cnn_mean_ms:.3f} ms ± {cnn_std_ms:.3f} ms  (device={DEVICE})')

CNN-5+6 extraction per image: 2.452 ms ± 0.523 ms  (device=cuda)


## Fuse Embeddings  (256 + 512 = 768-d)
CNN-5 (256-d) and CNN-6 (512-d) embeddings are concatenated along the feature axis  
to produce one 768-d vector per image.

In [6]:
# ================================================
# Cell 6 — Fuse Embeddings
# CNN-5 (256-d) + CNN-6 (512-d) = 768-d
# ================================================
# ── Feature fusion ─────────────────────────────────────────────
# Concatenate along feature axis: 256-d (CNN-5) + 512-d (CNN-6) = 768-d.
# Row alignment is guaranteed by shuffle=False in the DataLoaders.
# mRMR (Cell 7) will then select the most informative K* features,
# discarding redundant or low-relevance dimensions from either CNN.
XtrF = np.concatenate([emb5_tr, emb6_tr], axis=1)
XteF = np.concatenate([emb5_te, emb6_te], axis=1)
print(f'Fused train+val : {XtrF.shape}  |  Fused test: {XteF.shape}')
print(f'  CNN-5: 256-d  |  CNN-6: 512-d  |  Total: 768-d')

Fused train+val : (12000, 768)  |  Fused test: (3000, 768)
  CNN-5: 256-d  |  CNN-6: 512-d  |  Total: 768-d


## Two-Stage mRMR Feature Selector
**Stage 1 — MI pre-filter**: Mutual information scores all 768 features; keeps top 200 (fast, <30s).  
**Stage 2 — mRMR (MIQ)**: Runs on only those 200 features to select K* final features (~2-3 min).  
Without the pre-filter, mRMR on 768 features would take several hours.

In [7]:
# ================================================
# Cell 7 — MRMRSelector Definition
# Two-stage: MI pre-filter (768→200) then mRMR (200→K*)
# ================================================
# ── mRMR import ────────────────────────────────────────────────
# pymrmr wraps a C++ MIQ implementation — fast once compiled.
import pymrmr
# threading: background timer prints progress every 15s during mRMR
import threading


# Background timer thread — mRMR gives no internal progress bar;
# this prints elapsed time so it is clear the process is still running.
def _alive_printer(stop_event, start_time, interval=15):
    """Background thread: prints elapsed time every 15s so mRMR progress is visible."""
    while not stop_event.wait(interval):
        elapsed = time.perf_counter() - start_time
        print(f'       ... still running  ({elapsed:.0f}s elapsed) — please wait', flush=True)



# ── MRMRSelector — two-stage feature selector ──────────────────
# Stage 1 (MI pre-filter): scores all 768 features with mutual
#   information and keeps the top N_PRE=200.  Fast (<30s).
# Stage 2 (mRMR MIQ): selects K* from those 200 by maximising
#   relevance and minimising redundancy.  Runs in ~2-3 min.
# Without the pre-filter, mRMR on all 768 features takes hours.
class MRMRSelector(BaseEstimator, TransformerMixin):
    """
    Two-stage feature selector:
      Stage 1 — MI pre-filter : reduces d features → top N_PRE=200  (fast)
      Stage 2 — mRMR (MIQ)   : selects min(K*, N_PRE) from those 200
    N_PRE is always capped at 200 regardless of K*.
    """
    # Hard cap — keeps mRMR tractable regardless of K* setting
    N_PRE = 200   # hard cap — keeps mRMR tractable (~2-3 min)

    def __init__(self, k=128, n_bins=5, seed=42):
        self.k      = k
        self.n_bins = n_bins
        self.seed   = seed
        self.selected_indices_ = None

    def fit(self, X, y):
        t0_total = time.perf_counter()
        X     = np.asarray(X, dtype=np.float32)
        y     = np.asarray(y).astype(int)
        n_samples, n_features = X.shape

        n_pre = min(self.N_PRE, n_features)
        k_eff = min(self.k, n_pre)

        # Stage 1: MI pre-filter
        print(f'  [Stage 1/2] MI pre-filter  {n_features} → top {n_pre} features...', flush=True)
        t0 = time.perf_counter()
        mi_scores = mutual_info_classif(X, y, random_state=self.seed, n_neighbors=3)
        top_idx   = np.argsort(mi_scores)[::-1][:n_pre]
        X_pre     = X[:, top_idx]
        print(f'             done  ({time.perf_counter()-t0:.1f}s)', flush=True)

        # Stage 2: mRMR on top 200 features
        print(f'  [Stage 2/2] mRMR (select {k_eff} from {n_pre}, MIQ, n_bins={self.n_bins}) '
              f'— C++ call, no internal bar:', flush=True)
        disc = KBinsDiscretizer(n_bins=self.n_bins, encode='ordinal', strategy='quantile')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            X_disc = disc.fit_transform(X_pre)
        if hasattr(X_disc, 'toarray'): X_disc = X_disc.toarray()

        cols = [f'f{i}' for i in range(n_pre)]
        df   = pd.DataFrame(X_disc.astype(int), columns=cols)
        df.insert(0, 'class', y)

        # Background timer — shows that mRMR is alive
        stop_evt     = threading.Event()
        t_mrmr_start = time.perf_counter()
        timer        = threading.Thread(target=_alive_printer,
                                        args=(stop_evt, t_mrmr_start, 15),
                                        daemon=True)
        timer.start()
        # C++ mRMR call — blocks until complete; background timer shows progress
        selected = pymrmr.mRMR(df, 'MIQ', k_eff)
        stop_evt.set();  timer.join()
        print(f'             done  ({time.perf_counter()-t_mrmr_start:.1f}s)', flush=True)

        # Map selected column names back to original feature indices
        take = [n for n in selected if n.startswith('f')]
        idx_in_pre = []
        for n in take:
            try: idx_in_pre.append(int(n[1:]))
            except ValueError: pass
        if len(idx_in_pre) < k_eff:
            remaining = [i for i in range(n_pre) if i not in idx_in_pre]
            idx_in_pre.extend(remaining[:(k_eff - len(idx_in_pre))])

        self.selected_indices_ = [int(top_idx[i]) for i in idx_in_pre]
        print(f'  [mRMR] Total: {time.perf_counter()-t0_total:.1f}s  '
              f'| selected {len(self.selected_indices_)} features', flush=True)
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]

print('MRMRSelector ready  (N_PRE=200 hard cap, MI pre-filter → mRMR)')

MRMRSelector ready  (N_PRE=200 hard cap, MI pre-filter → mRMR)


## K* Selection — Mutual Information Elbow
Automatically selects how many features (K*) to keep by finding the elbow in the  
cumulative MI curve — the point of maximum gain relative to the chord.

In [8]:
# ================================================
# Cell 8 — K* Elbow Selection
# ================================================
# ── MI elbow method ────────────────────────────────────────────
# Finds K* = the point where cumulative MI deviates most from its
# chord (max perpendicular distance).  Avoids manually choosing K.
def pick_k_elbow(X, y, k_min=32, k_max=512, seed=SEED):
    print('Computing mutual information per feature...')
    Xs     = StandardScaler().fit_transform(X).astype(np.float32)
    mi     = mutual_info_classif(Xs, y, random_state=seed, n_neighbors=3)
    order  = np.argsort(mi)[::-1]
    cumsum = np.cumsum(mi[order])
    xvals  = np.arange(1, len(cumsum) + 1)
    chord  = np.linspace(cumsum[0], cumsum[-1], num=len(cumsum))
    k_auto = int(xvals[np.argmax(np.abs(cumsum - chord))])
    k_auto = int(np.clip(k_auto, k_min, k_max))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(xvals, cumsum,                  lw=2, label='Cumulative MI')
    ax.plot(xvals, chord,   '--', color='gray', lw=1, label='Chord')
    ax.axvline(k_auto, color='red', ls=':', lw=1.5, label=f'Elbow K*={k_auto}')
    ax.set_xlabel('Number of features (ranked by MI)', fontsize=11)
    ax.set_ylabel('Cumulative Mutual Information', fontsize=11)
    ax.set_title('K* Elbow Selection — CNN-5 + CNN-6 Fused Features', fontsize=12)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'k_elbow.png', dpi=150)
    plt.show()
    print(f'Auto K* = {k_auto}')
    return k_auto


# Run elbow on the full 768-d fused feature set
K_star = pick_k_elbow(XtrF, y_tr)
print(f'K* selected: {K_star}')

Computing mutual information per feature...
Auto K* = 370
K* selected: 370


/tmp/ipykernel_5915/2524624611.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ================================================
# Cell 9 — Confirm K* and Clear Stale Checkpoints
# K* is set by the MI elbow above.
# To override: set K_star = <integer> here.
# ================================================
print(f'Using K* = {K_star}')

# Clear stale checkpoints so tuning reruns with the new K*
# Remove outdated checkpoints so the tuning cells recompute
# from scratch whenever K* changes — prevents stale mRMR indices
# from being reused with a different feature count.
for stale in [CKPT_DIR / 'gamma_star.json', CKPT_DIR / 'C_star.json',
              CKPT_DIR / 'cv_results.json']:
    if stale.exists(): stale.unlink()
for f in CKPT_DIR.glob('mrmr_indices_K*.npy'):
    if f.name != f'mrmr_indices_K{K_star}.npy':
        f.unlink()

Using K* = 370


## Hyperparameter Tuning — gamma + C
Both mRMR and C-grid search run on a **stratified subsample of 3,000** training samples  
instead of all ~12,000 — this cuts tuning time by ~4× with minimal accuracy loss.  
Checkpoints are saved after each step so the cell can be safely re-run.

In [10]:
# ================================================
# Cell 10 — mRMR Feature Selection + Gamma + C Tuning
# All three steps are checkpointed and resumable.
# ================================================
# ── RBF gamma estimation ───────────────────────────────────────
# gamma* = 1 / (2 * median_pairwise_distance^2) on a random
# subsample.  More principled than sklearn's default 'scale'
# heuristic for high-dimensional fused embeddings.
def median_gamma(X, max_samples=500, seed=SEED):
    """Estimate RBF gamma as 1 / (2 * median_distance^2)."""
    rng   = np.random.RandomState(seed)
    idx   = rng.choice(len(X), size=min(len(X), max_samples), replace=False)
    dists = pairwise_distances(X[idx], metric='euclidean', n_jobs=-1)
    iu    = np.triu_indices_from(dists, k=1)
    med   = np.median(dists[iu])
    if med <= 1e-12 or not np.isfinite(med): return 'scale'
    return float(1.0 / (2.0 * med ** 2))



# ── Checkpoint paths ───────────────────────────────────────────
# Each step (mRMR, gamma, C) is checkpointed so the cell can be
# safely interrupted and re-run without restarting from scratch.
MRMR_IDX_PATH   = CKPT_DIR / f'mrmr_indices_K{K_star}.npy'
GAMMA_PATH      = CKPT_DIR / 'gamma_star.json'
CV_RESULTS_PATH = CKPT_DIR / 'cv_results.json'
# Subsampling: mRMR and C-search use 3,000 stratified samples
# instead of all 12,000 — ~4x faster with minimal accuracy loss.
TUNE_SUBSAMPLE  = 3000   # subsample size for mRMR and C tuning

# ── Step 1: MI pre-filter + mRMR on stratified subsample ─────────────────

# ── Step 1: MI pre-filter + mRMR (resumable) ──────────────────
if MRMR_IDX_PATH.exists() and GAMMA_PATH.exists():
    mrmr_indices = np.load(MRMR_IDX_PATH)
    with open(GAMMA_PATH) as f: gamma_star = json.load(f)['gamma_star']
    print(f'Resumed mRMR indices + gamma* = {gamma_star:.6g}')
else:
    if len(y_tr) > TUNE_SUBSAMPLE:
        from sklearn.model_selection import StratifiedShuffleSplit
        sss_m = StratifiedShuffleSplit(n_splits=1, train_size=TUNE_SUBSAMPLE, random_state=SEED)
        m_idx, _ = next(sss_m.split(XtrF, y_tr))
        XtrF_m, y_m = XtrF[m_idx], y_tr[m_idx]
        print(f'Subsample: {TUNE_SUBSAMPLE} / {len(y_tr)} samples (~{TUNE_SUBSAMPLE//len(np.unique(y_tr))}/class)')
    else:
        XtrF_m, y_m = XtrF, y_tr

    print(f'Running MI + mRMR (K={K_star}) on {len(y_m)} × {XtrF_m.shape[1]}...')
    sc_tmp  = StandardScaler().fit(XtrF_m)
    sel_tmp = MRMRSelector(k=K_star, n_bins=5, seed=SEED)
    sel_tmp.fit(sc_tmp.transform(XtrF_m), y_m)
    mrmr_indices = np.array(sel_tmp.selected_indices_)
    np.save(MRMR_IDX_PATH, mrmr_indices)

    X_sel      = sc_tmp.transform(XtrF_m)[:, mrmr_indices]
    gamma_star = median_gamma(X_sel)
    with open(GAMMA_PATH, 'w') as f: json.dump({'gamma_star': gamma_star}, f)
    print(f'Done. gamma* = {gamma_star:.6g}')

# Feature provenance: CNN-5 features are indices 0–255, CNN-6 are 256–767

# ── Feature provenance ──────────────────────────────────────────
# CNN-5 features occupy indices 0-255 in the 768-d fused vector;
# CNN-6 features occupy indices 256-767.
from_cnn5 = int(np.sum(np.array(mrmr_indices) < 256))
from_cnn6 = int(np.sum(np.array(mrmr_indices) >= 256))
print(f'\nFeature provenance (K={K_star}):')
print(f'  CNN-5 contributed: {from_cnn5} features ({from_cnn5/K_star*100:.1f}%)')
print(f'  CNN-6 contributed: {from_cnn6} features ({from_cnn6/K_star*100:.1f}%)')


# Apply mRMR selection to both train+val and test sets
Xtr_sel = XtrF[:, mrmr_indices]
Xte_sel = XteF[:, mrmr_indices]

# ── Step 2: Tune C on stratified subsample ────────────────────────────────

# ── Step 2: C hyperparameter search (3-fold CV, resumable) ───
# class_weight='balanced' compensates for any class imbalance.
# max_iter=5000 prevents convergence warnings on the subsample.
C_PATH = CKPT_DIR / 'C_star.json'
C_GRID = (1, 10, 30, 100, 500)

if C_PATH.exists():
    with open(C_PATH) as f: d = json.load(f)
    C_star, cv_f1 = d['C_star'], d['cv_f1']
    if CV_RESULTS_PATH.exists():
        with open(CV_RESULTS_PATH) as f: cv_rows = json.load(f)
    else:
        cv_rows = [{'C': C_star, 'mean_macro_F1': cv_f1}]
    print(f'\nResumed C* = {C_star}  (cv macro-F1 = {cv_f1:.4f})')
else:
    if len(y_tr) > TUNE_SUBSAMPLE:
        from sklearn.model_selection import StratifiedShuffleSplit
        sss_c = StratifiedShuffleSplit(n_splits=1, train_size=TUNE_SUBSAMPLE, random_state=SEED+1)
        c_idx, _ = next(sss_c.split(Xtr_sel, y_tr))
        Xcv, ycv = Xtr_sel[c_idx], y_tr[c_idx]
        print(f'CV subsample: {TUNE_SUBSAMPLE} / {len(y_tr)} samples')
    else:
        Xcv, ycv = Xtr_sel, y_tr

    print(f'\n3-fold CV over C in {C_GRID}...')
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    best_C, best_f1, cv_rows = None, -1.0, []

    for C in tqdm(C_GRID, desc='C candidates', unit='C'):
        t_c    = time.perf_counter()
        scores = []
        fold_iter = tqdm(enumerate(skf.split(Xcv, ycv), 1),
                         total=3, desc=f'  C={C}', leave=False, unit='fold')
        for fold_n, (tr_i, va_i) in fold_iter:
            sc  = StandardScaler().fit(Xcv[tr_i])
            clf = SVC(kernel='rbf', class_weight='balanced',
                      C=C, gamma=gamma_star, cache_size=2000, max_iter=5000)
            clf.fit(sc.transform(Xcv[tr_i]), ycv[tr_i])
            fold_f1 = f1_score(ycv[va_i],
                               clf.predict(sc.transform(Xcv[va_i])),
                               average='macro')
            scores.append(fold_f1)
            fold_iter.set_postfix(f1=f'{fold_f1:.4f}')

        mean_f1 = float(np.mean(scores))
        elapsed = time.perf_counter() - t_c
        tqdm.write(f'  C={C:<5}  macro-F1={mean_f1:.4f}  ({elapsed:.1f}s)')
        cv_rows.append({'C': C, 'mean_macro_F1': round(mean_f1, 4)})
        if mean_f1 > best_f1: best_f1, best_C = mean_f1, C

    C_star, cv_f1 = best_C, best_f1
    with open(C_PATH,          'w') as f: json.dump({'C_star': C_star, 'cv_f1': cv_f1}, f)
    with open(CV_RESULTS_PATH, 'w') as f: json.dump(cv_rows, f, indent=2)
    print('\nCV results:')
    print(pd.DataFrame(cv_rows).to_string(index=False))

print(f'\nK*={K_star}  |  C*={C_star}  |  gamma*={gamma_star}')

Subsample: 3000 / 12000 samples (~500/class)
Running MI + mRMR (K=370) on 3000 × 768...
  [Stage 1/2] MI pre-filter  768 → top 200 features...
             done  (6.8s)
  [Stage 2/2] mRMR (select 200 from 200, MIQ, n_bins=5) — C++ call, no internal bar:
       ... still running  (566s elapsed) — please wait


 *** This program and the respective minimum Redundancy Maximum Relevance (mRMR) 
     algorithm were developed by Hanchuan Peng <hanchuan.peng@gmail.com>for
     the paper 
     "Feature selection based on mutual information: criteria of 
      max-dependency, max-relevance, and min-redundancy,"
      Hanchuan Peng, Fuhui Long, and Chris Ding, 
      IEEE Transactions on Pattern Analysis and Machine Intelligence,
      Vol. 27, No. 8, pp.1226-1238, 2005.


*** MaxRel features ***
Order 	 Fea 	 Name 	 Score
1 	 2 	 f1 	 1.095
2 	 15 	 f14 	 1.073
3 	 1 	 f0 	 1.067
4 	 10 	 f9 	 1.062
5 	 7 	 f6 	 1.047
6 	 6 	 f5 	 1.047
7 	 9 	 f8 	 1.041
8 	 12 	 f11 	 1.035
9 	 3 	 f2 	 1.033


C candidates:   0%|          | 0/5 [00:00<?, ?C/s]

  C=1:   0%|          | 0/3 [00:00<?, ?fold/s]

  C=1      macro-F1=0.8453  (0.5s)


  C=10:   0%|          | 0/3 [00:00<?, ?fold/s]

...

  C=100    macro-F1=0.8400  (0.5s)


  C=500:   0%|          | 0/3 [00:00<?, ?fold/s]

/home/jenarththan/Desktop/FYP/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/jenarththan/Desktop/FYP/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  C=500    macro-F1=0.8303  (0.6s)

CV results:
  C  mean_macro_F1
  1         0.8453
 10         0.8457
 30         0.8457
100         0.8400
500         0.8303

K*=370  |  C*=10  |  gamma*=0.0014115454396232963


/home/jenarththan/Desktop/FYP/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [11]:
# ================================================
# Cell 11 — C Hyperparameter Search Chart
# ================================================
# ── C search bar chart ─────────────────────────────────────────
# Highlights the best C in orange; others in blue.
cv_df  = pd.DataFrame(cv_rows)
colors = ['darkorange' if c == C_star else 'steelblue' for c in cv_df['C']]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(cv_df)), cv_df['mean_macro_F1'],
              color=colors, edgecolor='white', linewidth=0.6)
ax.set_xticks(range(len(cv_df)))
ax.set_xticklabels([f'C = {c}' for c in cv_df['C']], fontsize=11)
ax.set_ylabel('3-fold CV Macro-F1', fontsize=12)
ax.set_title(f'SVM C Hyperparameter Search  (best: C = {C_star})', fontsize=13)
ylo = max(0,   cv_df['mean_macro_F1'].min() - 0.05)
yhi = min(1.0, cv_df['mean_macro_F1'].max() + 0.06)
ax.set_ylim(ylo, yhi)
for bar, val in zip(bars, cv_df['mean_macro_F1']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='darkorange', label=f'Best  C = {C_star}'),
    plt.Rectangle((0, 0), 1, 1, color='steelblue',  label='Other candidates'),
], fontsize=10)
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cv_C_search.png', dpi=150)
plt.show()
print('Saved → Plots/cv_C_search.png')
pd.DataFrame(cv_rows).to_csv(REPS_DIR / 'cv_results.csv', index=False)
print('Saved → Reports/cv_results.csv')

Saved → Plots/cv_C_search.png
Saved → Reports/cv_results.csv


/tmp/ipykernel_5915/2328272564.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final SVM Fit + Evaluate
Final SVM is trained on the full train+val set (12,000 images) using the best K*, C*, gamma*.  
Evaluation is done on the held-out test set (3,000 images).

In [12]:
# ================================================
# Cell 12 — Final SVM Fit + Test Evaluation
# ================================================
# ── Additional metrics for ROC and PR curves ───────────────────
from sklearn.metrics import (roc_curve, auc,
                              precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize


# ── Final SVM — fit on full train+val (12,000 samples) ────────
# StandardScaler is fitted on train+val only (no test data leakage).
# cache_size=2000 MB speeds up RBF kernel matrix computation.
scaler_final = StandardScaler().fit(Xtr_sel)
clf_final    = SVC(kernel='rbf', class_weight='balanced',
                   C=C_star, gamma=gamma_star, cache_size=2000, max_iter=10000)

gamma_str = f'{gamma_star:.4g}' if isinstance(gamma_star, float) else gamma_star
print(f'Fitting SVC on {len(y_tr)} samples  (K*={K_star}, C={C_star}, gamma={gamma_str})...')
t0 = time.perf_counter()
clf_final.fit(scaler_final.transform(Xtr_sel), y_tr)
fit_time = time.perf_counter() - t0
print(f'Done in {fit_time:.1f}s  |  support vectors: {clf_final.n_support_.sum()} total  {clf_final.n_support_.tolist()}')

print('\nPredicting on test set...')
Xte_sc   = scaler_final.transform(Xte_sel)
y_pred   = clf_final.predict(Xte_sc)
y_score  = clf_final.decision_function(Xte_sc)     # (n_test, n_classes) OvR scores
y_te_bin = label_binarize(y_te, classes=range(NUM_CLASSES))


# ── Test metrics ───────────────────────────────────────────────
acc  = accuracy_score(y_te, y_pred)
f1M  = f1_score(y_te, y_pred, average='macro')
f1m  = f1_score(y_te, y_pred, average='micro')
prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_te, y_pred,    average='macro', zero_division=0)


# ── Per-class AUC-ROC and Average Precision (OvR) ─────────────
# decision_function returns OvR scores — used directly for ROC/PR.
roc_auc  = {}
avg_prec = {}
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
    roc_auc[cls]  = round(float(auc(fpr, tpr)), 4)
    avg_prec[cls] = round(float(average_precision_score(y_te_bin[:, i], y_score[:, i])), 4)

print(f'\n=== Test Results  (K*={K_star}, C*={C_star}) ===')
print(f'  Accuracy          : {acc:.4f}')
print(f'  Macro-F1          : {f1M:.4f}')
print(f'  Micro-F1          : {f1m:.4f}')
print(f'  Macro-Precision   : {prec:.4f}')
print(f'  Macro-Recall      : {rec:.4f}')
print(f'  Mean AUC-ROC      : {np.mean(list(roc_auc.values())):.4f}')
print(f'  Mean Avg-Precision: {np.mean(list(avg_prec.values())):.4f}')
print(f'\n  Per-class AUC-ROC : {roc_auc}')
print(f'  Per-class Avg-Prec: {avg_prec}')


# ── Classification report ──────────────────────────────────────
# Saved as both .txt (human-readable) and .csv (for analysis).
rep_str  = classification_report(y_te, y_pred, target_names=CLASS_NAMES, digits=4)
rep_dict = classification_report(y_te, y_pred, target_names=CLASS_NAMES,
                                  digits=4, output_dict=True)
print('\n' + rep_str)
with open(REPS_DIR / 'classification_report.txt', 'w') as f: f.write(rep_str)
pd.DataFrame(rep_dict).transpose().to_csv(REPS_DIR / 'classification_report.csv')
print('Saved → Reports/')

Fitting SVC on 12000 samples  (K*=370, C=10, gamma=0.001412)...
Done in 1.3s  |  support vectors: 3932 total  [395, 217, 663, 1184, 1264, 209]

Predicting on test set...

=== Test Results  (K*=370, C*=10) ===
  Accuracy          : 0.8247
  Macro-F1          : 0.8235
  Micro-F1          : 0.8247
  Macro-Precision   : 0.8270
  Macro-Recall      : 0.8247
  Mean AUC-ROC      : 0.9690
  Mean Avg-Precision: 0.8673

  Per-class AUC-ROC : {'0': 0.9854, '100': 0.9963, '500': 0.9689, '1000': 0.9381, '1500': 0.9299, '2000': 0.9955}
  Per-class Avg-Prec: {'0': 0.9408, '100': 0.9812, '500': 0.8842, '1000': 0.7489, '1500': 0.6665, '2000': 0.9821}

              precision    recall  f1-score   support

           0     0.8687    0.9000    0.8841       500
         100     0.9425    0.9500    0.9462       500
         500     0.8004    0.8100    0.8052       500
        1000     0.7577    0.5940    0.6659       500
        1500     0.6432    0.7500    0.6925       500
        2000     0.9497    0.9440

## Full Pipeline Inference Time Per Image
Measures total time: CNN-5+6 embedding extraction + mRMR feature selection + SVM prediction.

In [13]:
# ================================================
# Cell 13 — Full Pipeline Inference Time
# ================================================
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# Warm-up: flush GPU and SVM caches before the timed runs
for _ in tqdm(range(WARMUP), desc='Warmup', leave=False):
    with torch.no_grad():
        _, e5 = cnn5(dummy, return_embedding=True)
        _, e6 = cnn6(dummy, return_embedding=True)
    fused = np.concatenate([e5.cpu().numpy(), e6.cpu().numpy()], axis=1)
    clf_final.predict(scaler_final.transform(fused[:, mrmr_indices]))


# ── Split-stage timing ─────────────────────────────────────────
# t0->t1: CNN-5+6 extraction only
# t0->t2: full pipeline (extraction + mRMR masking + SVM predict)
# Difference gives the SVM overhead in isolation.
cnn_ms, full_ms = [], []
for _ in tqdm(range(N_RUNS), desc='Timing runs', unit='img'):
    t0 = time.perf_counter()
    with torch.no_grad():
        _, e5 = cnn5(dummy, return_embedding=True)
        _, e6 = cnn6(dummy, return_embedding=True)
    t1 = time.perf_counter()
    fused = np.concatenate([e5.cpu().numpy(), e6.cpu().numpy()], axis=1)
    clf_final.predict(scaler_final.transform(fused[:, mrmr_indices]))
    t2 = time.perf_counter()
    cnn_ms.append((t1 - t0) * 1000)
    full_ms.append((t2 - t0) * 1000)

cnn_mean_ms  = float(np.mean(cnn_ms));  cnn_std_ms  = float(np.std(cnn_ms))
full_mean_ms = float(np.mean(full_ms)); full_std_ms = float(np.std(full_ms))
svm_diffs    = [f - c for f, c in zip(full_ms, cnn_ms)]
svm_mean_ms  = float(np.mean(svm_diffs))
svm_std_ms   = float(np.std(svm_diffs))

print(f'Per-image inference time  (device={DEVICE}, {N_RUNS} runs):')
print(f'  CNN-5+6 extraction : {cnn_mean_ms:.3f} ms ± {cnn_std_ms:.3f} ms')
print(f'  SVM prediction     : {svm_mean_ms:.3f} ms ± {svm_std_ms:.3f} ms')
print(f'  Full pipeline      : {full_mean_ms:.3f} ms ± {full_std_ms:.3f} ms')


# Persist timing breakdown for the thesis inference table
timing_report = {
    'device'                : str(DEVICE),
    'n_runs'                : N_RUNS,
    'cnn_extraction_mean_ms': round(cnn_mean_ms,  4),
    'cnn_extraction_std_ms' : round(cnn_std_ms,   4),
    'svm_predict_mean_ms'   : round(svm_mean_ms,  4),
    'svm_predict_std_ms'    : round(svm_std_ms,   4),
    'full_pipeline_mean_ms' : round(full_mean_ms, 4),
    'full_pipeline_std_ms'  : round(full_std_ms,  4),
}
with open(REPS_DIR / 'inference_time.json', 'w') as f:
    json.dump(timing_report, f, indent=2)
print('Saved → Reports/inference_time.json')

Warmup:   0%|          | 0/20 [00:00<?, ?it/s]

Timing runs:   0%|          | 0/200 [00:00<?, ?img/s]

Per-image inference time  (device=cuda, 200 runs):
  CNN-5+6 extraction : 1.034 ms ± 0.155 ms
  SVM prediction     : 2.559 ms ± 0.462 ms
  Full pipeline      : 3.594 ms ± 0.501 ms
Saved → Reports/inference_time.json


## Confusion Matrix, Per-Class Metrics & UMAP

In [14]:
# ================================================
# Cell 14 — Visualisations
# 1. Confusion matrix (counts)
# 2. Confusion matrix (normalised %)
# 3. Per-class Precision / Recall / F1 bar chart
# 4. ROC curves
# 5. Precision-Recall curves
# 6. Feature provenance (CNN-5 vs CNN-6)
# 7. Inference time breakdown
# 8. UMAP of train+val features
# ================================================
# ── Per-class scores ───────────────────────────────────────────
prec_c = precision_score(y_te, y_pred, average=None, labels=range(NUM_CLASSES), zero_division=0)
rec_c  = recall_score(y_te,  y_pred,   average=None, labels=range(NUM_CLASSES), zero_division=0)
f1_c   = f1_score(y_te,     y_pred,    average=None, labels=range(NUM_CLASSES), zero_division=0)

COLORS = plt.cm.tab10(np.linspace(0, 0.9, NUM_CLASSES))

# ── 1. Confusion Matrix (counts) ──────────────────────────────────────────

# ── 1. Confusion matrix (raw counts) ───────────────────────────
cm = confusion_matrix(y_te, y_pred, labels=list(range(NUM_CLASSES)))

# ── 4. ROC curves (one-vs-rest) ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(
    cmap='Blues', values_format='d', ax=ax)
ax.set_title(f'Confusion Matrix — CNN5+CNN6-MI-SVM  (K*={K_star}, C*={C_star})', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved: Plots/confusion_matrix.png')

# ── 2. Normalised Confusion Matrix (%) ───────────────────────────────────

# ── 2. Normalised confusion matrix (%) ────────────────────────
# Row-normalised: shows per-class recall directly on the diagonal.
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

# ── 4. ROC curves (one-vs-rest) ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES).plot(
    cmap='Blues', values_format='.1f', ax=ax)
ax.set_title(f'Normalised Confusion Matrix (%) — CNN5+CNN6-MI-SVM  (K*={K_star})', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix_normalized.png', dpi=150)
plt.show()
print('Saved: Plots/confusion_matrix_normalized.png')

# ── 3. Per-Class Precision / Recall / F1 ─────────────────────────────────

# ── 3. Per-class Precision / Recall / F1 bar chart ────────────
x, w = np.arange(NUM_CLASSES), 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, prec_c, w, label='Precision', color='#4C72B0')
ax.bar(x,     rec_c,  w, label='Recall',    color='#DD8452')
ax.bar(x + w, f1_c,   w, label='F1-Score',  color='#55A868')
for xi, (p, r, f) in enumerate(zip(prec_c, rec_c, f1_c)):
    ax.text(xi - w, p + 0.012, f'{p:.2f}', ha='center', va='bottom', fontsize=8)
    ax.text(xi,     r + 0.012, f'{r:.2f}', ha='center', va='bottom', fontsize=8)
    ax.text(xi + w, f + 0.012, f'{f:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([f'{c} RPM' for c in CLASS_NAMES], fontsize=11)
ax.set_ylim(0, 1.12); ax.legend(fontsize=11)
ax.grid(axis='y', ls='--', alpha=0.4)
ax.set_title(f'Per-Class Metrics — CNN5+CNN6-MI-SVM  (K*={K_star}, C*={C_star})', fontsize=12)
ax.set_xlabel('Milling Class (RPM)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'per_class_metrics.png', dpi=150)
plt.show()
print('Saved: Plots/per_class_metrics.png')

# ── 4. ROC Curves ────────────────────────────────────────────────────────

# ── 4. ROC curves (one-vs-rest) ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
    ax.plot(fpr, tpr, lw=2, color=COLORS[i],
            label=f'{cls} RPM  (AUC = {roc_auc[cls]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random  (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ROC Curves (OvR) — Mean AUC = {np.mean(list(roc_auc.values())):.3f}', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_curves.png', dpi=150)
plt.show()
print('Saved: Plots/roc_curves.png')

# ── 5. Precision-Recall Curves ───────────────────────────────────────────

# ── 4. ROC curves (one-vs-rest) ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(CLASS_NAMES):
    p_c, r_c, _ = precision_recall_curve(y_te_bin[:, i], y_score[:, i])
    ax.plot(r_c, p_c, lw=2, color=COLORS[i],
            label=f'{cls} RPM  (AP = {avg_prec[cls]:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curves (OvR) — Mean AP = {np.mean(list(avg_prec.values())):.3f}',
             fontsize=13)
ax.legend(fontsize=10, loc='lower left')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'precision_recall_curves.png', dpi=150)
plt.show()
print('Saved: Plots/precision_recall_curves.png')

# ── 6. Feature Provenance (CNN-5 vs CNN-6) ───────────────────────────────
fp_labels = ['CNN-5\n(256-d)', 'CNN-6\n(512-d)']
fp_counts = [from_cnn5, from_cnn6]
fp_colors = ['#DD8452', '#55A868']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(fp_labels, fp_counts, color=fp_colors, edgecolor='white')
for xi, v in enumerate(fp_counts):
    axes[0].text(xi, v + 0.3, str(v), ha='center', va='bottom',
                 fontsize=13, fontweight='bold')
axes[0].set_ylabel('Features Selected by mRMR', fontsize=11)
axes[0].set_title(f'Feature Count per CNN  (K* = {K_star})', fontsize=12)
axes[0].grid(axis='y', ls='--', alpha=0.4)
axes[1].pie(fp_counts, labels=fp_labels, colors=fp_colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 12})
axes[1].set_title('Relative Feature Contribution (%)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'feature_provenance.png', dpi=150)
plt.show()
print('Saved: Plots/feature_provenance.png')

# ── 7. Inference Time Breakdown ───────────────────────────────────────────
stages = ['CNN-5+6\nExtraction', 'SVM\nPrediction', 'Full\nPipeline']
means  = [cnn_mean_ms, svm_mean_ms, full_mean_ms]
stds   = [cnn_std_ms,  svm_std_ms,  full_std_ms]
tcols  = ['#4C72B0', '#55A868', '#C44E52']
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(stages, means, yerr=stds, color=tcols, capsize=6,
              edgecolor='white', linewidth=0.5,
              error_kw=dict(elinewidth=1.5, capthick=1.5))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + 0.05,
            f'{m:.2f} ± {s:.2f} ms', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title(f'Per-Image Inference Time  (device={DEVICE}, N={N_RUNS})', fontsize=12)
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'inference_time_breakdown.png', dpi=150)
plt.show()
print('Saved: Plots/inference_time_breakdown.png')

# ── 8. UMAP ──────────────────────────────────────────────────────────────
try:

    # ── 8. UMAP 2-D projection ─────────────────────────────────────
    # Skipped gracefully if umap-learn is not installed.
    print('Running UMAP (may take ~1-2 min)...')
    Xt  = scaler_final.transform(Xtr_sel)
    X2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(Xt)
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        mask = (y_tr == i)
        ax.scatter(X2d[mask, 0], X2d[mask, 1], s=10,
                   color=COLORS[i], label=f'{cls} RPM', alpha=0.7)
    ax.legend(markerscale=2, fontsize=10, title='Milling Class', title_fontsize=10)
    ax.set_title(f'UMAP — mRMR-Selected Features (K*={K_star}, Train+Val)', fontsize=12)
    ax.set_xlabel('UMAP-1', fontsize=11); ax.set_ylabel('UMAP-2', fontsize=11)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'umap_train.png', dpi=150)
    plt.show()
    print('Saved: Plots/umap_train.png')
except Exception as e:
    print(f'UMAP skipped: {e}')

/tmp/ipykernel_5915/2570395357.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_5915/2570395357.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: Plots/confusion_matrix.png
Saved: Plots/confusion_matrix_normalized.png


/tmp/ipykernel_5915/2570395357.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_5915/2570395357.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: Plots/per_class_metrics.png
Saved: Plots/roc_curves.png


/tmp/ipykernel_5915/2570395357.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_5915/2570395357.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: Plots/precision_recall_curves.png
Saved: Plots/feature_provenance.png


/tmp/ipykernel_5915/2570395357.py:134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/jenarththan/Desktop/FYP/venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: Plots/inference_time_breakdown.png
Running UMAP (may take ~1-2 min)...
Saved: Plots/umap_train.png


/tmp/ipykernel_5915/2570395357.py:152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save All Results

In [15]:
# ================================================
# Cell 15 — Save Pipeline + Reports + Summary
# ================================================
# ── Save SVM pipeline ──────────────────────────────────────────
# Bundles scaler + clf + mrmr_indices in one file so inference
# only needs to load this artefact alongside the CNN checkpoints.
joblib.dump({'scaler': scaler_final, 'clf': clf_final, 'mrmr_indices': mrmr_indices},
            MDLS_DIR / 'svm_pipeline.joblib')
print('Pipeline saved → Models/svm_pipeline.joblib')


# ── Per-class metrics CSV ──────────────────────────────────────
pd.DataFrame([{
    'Class'         : cls,
    'Precision'     : round(float(prec_c[i]), 4),
    'Recall'        : round(float(rec_c[i]),  4),
    'F1'            : round(float(f1_c[i]),   4),
    'AUC_ROC'       : roc_auc[cls],
    'Avg_Precision' : avg_prec[cls],
    'Support'       : int(np.sum(y_te == i)),
} for i, cls in enumerate(CLASS_NAMES)]).to_csv(REPS_DIR / 'per_class_metrics.csv', index=False)


# ── Feature provenance CSV ────────────────────────────────────
# Records per-CNN feature contribution for the thesis discussion.
pd.DataFrame([
    {'CNN': 'CNN-5', 'Embed_Dim': 256, 'Features_Selected': from_cnn5,
     'Pct': round(from_cnn5/K_star*100, 1)},
    {'CNN': 'CNN-6', 'Embed_Dim': 512, 'Features_Selected': from_cnn6,
     'Pct': round(from_cnn6/K_star*100, 1)},
]).to_csv(REPS_DIR / 'feature_provenance.csv', index=False)


# ── Final summary JSON + CSV ───────────────────────────────────
# Single file with all hyperparameters, metrics, and timing —
# sufficient to fully reproduce the reported results.
gamma_val = float(gamma_star) if isinstance(gamma_star, float) else gamma_star
summary = {
    'dataset'              : str(BASE / 'May11/Dataset'),
    'cnns_used'            : 'CNN-5 + CNN-6',
    'train_samples'        : int(len(y_train)),
    'val_samples'          : int(len(y_val)),
    'fit_samples'          : int(len(y_tr)),
    'test_samples'         : int(len(y_te)),
    'fused_dim'            : int(XtrF.shape[1]),
    'K_star'               : int(K_star),
    'C_star'               : float(C_star),
    'gamma_star'           : gamma_val,
    'cv_macro_f1'          : round(float(cv_f1), 4),
    'test_accuracy'        : round(acc,  4),
    'test_macro_f1'        : round(f1M,  4),
    'test_micro_f1'        : round(f1m,  4),
    'test_macro_precision' : round(prec, 4),
    'test_macro_recall'    : round(rec,  4),
    'mean_auc_roc'         : round(float(np.mean(list(roc_auc.values()))),  4),
    'mean_avg_precision'   : round(float(np.mean(list(avg_prec.values()))), 4),
    'features_from_cnn5'   : from_cnn5,
    'features_from_cnn6'   : from_cnn6,
    **timing_report,
}
with open(REPS_DIR / 'final_summary.json', 'w') as f:
    json.dump({**summary,
               'per_class_auc_roc'      : roc_auc,
               'per_class_avg_precision': avg_prec}, f, indent=2)
pd.DataFrame([summary]).to_csv(REPS_DIR / 'final_summary.csv', index=False)

gamma_display = f'{gamma_star:.6g}' if isinstance(gamma_star, float) else gamma_star
print('\n' + '='*65)
print('  FINAL SUMMARY  (CNN-5 + CNN-6 → MI-Filter → mRMR → SVM)')
print('='*65)
print(f'  Train / Val / Test  : {len(y_train)} / {len(y_val)} / {len(y_te)}')
print(f'  Fused dim → K*      : {XtrF.shape[1]} → {K_star}')
print(f'  C* / gamma*         : {C_star} / {gamma_display}')
print(f'  Test Accuracy       : {acc:.4f}')
print(f'  Test Macro-F1       : {f1M:.4f}')
print(f'  Mean AUC-ROC        : {np.mean(list(roc_auc.values())):.4f}')
print(f'  CNN infer/img       : {cnn_mean_ms:.3f} ms ± {cnn_std_ms:.3f} ms')
print(f'  Full pipeline/img   : {full_mean_ms:.3f} ms ± {full_std_ms:.3f} ms')
print(f'  Feature provenance  : CNN-5={from_cnn5} | CNN-6={from_cnn6}')
print('='*65)
print(f'\nAll results → {OUT_DIR}')
print('\nSaved plots:')
for p in sorted(PLOTS_DIR.glob('*.png')): print(f'  Plots/{p.name}')
print('Saved reports:')
for p in sorted(REPS_DIR.glob('*')):     print(f'  Reports/{p.name}')

Pipeline saved → Models/svm_pipeline.joblib

  FINAL SUMMARY  (CNN-5 + CNN-6 → MI-Filter → mRMR → SVM)
  Train / Val / Test  : 10500 / 1500 / 3000
  Fused dim → K*      : 768 → 370
  C* / gamma*         : 10 / 0.00141155
  Test Accuracy       : 0.8247
  Test Macro-F1       : 0.8235
  Mean AUC-ROC        : 0.9690
  CNN infer/img       : 1.034 ms ± 0.155 ms
  Full pipeline/img   : 3.594 ms ± 0.501 ms
  Feature provenance  : CNN-5=64 | CNN-6=136

All results → /home/jenarththan/Desktop/FYP/May11/Notebooks/Custom/Hybrid56_May11_Results

Saved plots:
  Plots/confusion_matrix.png
  Plots/confusion_matrix_normalized.png
  Plots/cv_C_search.png
  Plots/feature_provenance.png
  Plots/inference_time_breakdown.png
  Plots/k_elbow.png
  Plots/per_class_metrics.png
  Plots/precision_recall_curves.png
  Plots/roc_curves.png
  Plots/umap_train.png
Saved reports:
  Reports/classification_report.csv
  Reports/classification_report.txt
  Reports/cv_results.csv
  Reports/feature_provenance.csv
  Reports/